# Analysis for asymmetric difference

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import re

# ==========================================
# 設定
# ==========================================
RESULTS_DIR = "./results"       # データのルートディレクトリ
WINDOW_SIZE = 100               # 平滑化のウィンドウサイズ
MAX_INDEX = 40000                # 読み込む最大ステップ数（Noneの場合はあるだけ読む）
BINS = ["bin_0", "bin_1", "bin_2", "bin_3", "bin_4"] # カラム名定義
TARGET_SIGN = -1.0

# =========================================
# seedに基づいて実験結果ディレクトリを収集
# =========================================

all_dfs = []
run_dirs = glob.glob(os.path.join(RESULTS_DIR, "run_*"))
run_dirs = sorted(run_dirs)

# 特定のシード範囲に限定したい場合
# ========================================
SEED_TABLE = [10, 11, 12, 13, 14, 16, 17, 18, 19, 20, 21]
run_dirs = [
    os.path.join(RESULTS_DIR, f"run_{seed}_dir_{TARGET_SIGN}")
    for seed in SEED_TABLE
    if os.path.isdir(os.path.join(RESULTS_DIR, f"run_{seed}_dir_{TARGET_SIGN}"))
]
# =========================================

# 出力先
OUTPUT_DIR = "./results/LR_diff_summary"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import os
import glob
import re
import networkx as nx
from collections import defaultdict

def identify_core(seed, target_sign): 
    """
    ある1つのrunの結果ファイルに基づいて、全エージェントをcore or noncoreに分類する。
    - 全ステップで {0, 1} のみ -> minus-core
    - 全ステップで {3, 4} のみ -> plus-core
    - それ以外 (2を含む、あるいは {0,1} と {3,4} を行き来する) -> noncore
    """
    
    search_pattern = os.path.join(RESULTS_DIR, f"run_{seed}_dir_{target_sign}", "GEXF", "lambda_0.0", "*.gexf")
    files = glob.glob(search_pattern)
    
    if not files:
        print(f"Warning: No GEXF files found for run_{seed}_dir_{target_sign}")
        return {}

    # ステップ数を抽出してソートするための辞書を作成
    step_file_map = {}
    for f in files:
        match = re.search(r"step_(\d+)\.gexf", os.path.basename(f))
        if match:
            step = int(match.group(1))
            step_file_map[step] = f
    
    sorted_steps = sorted(step_file_map.keys())
    # MAX_INDEX が定義されている場合はフィルタリング (環境に合わせて調整してください)
    if 'MAX_INDEX' in globals() and MAX_INDEX is not None:
        sorted_steps = [s for s in sorted_steps if s <= MAX_INDEX]

    # 各エージェントが経験した opinionclass をすべて記録するセット
    # { "node_0": {0, 1}, "node_1": {2, 3}, ... }
    agent_classes_history = defaultdict(set)

    # 各ステップのファイルを読み込み
    print(f"Processing {len(sorted_steps)} steps for seed {seed}...")
    for step in sorted_steps:
        gexf_path = step_file_map[step]
        try:
            G = nx.read_gexf(gexf_path)
            
            for node, data in G.nodes(data=True):
                # 属性キーの大文字・小文字揺れを吸収して opinionclass を取得
                op_class = None
                for key, val in data.items():
                    if str(key).lower() == 'opinionclass':
                        try:
                            op_class = int(float(val))
                        except (ValueError, TypeError):
                            pass
                        break
                
                if op_class is not None:
                    agent_classes_history[node].add(op_class)
                    
        except Exception as e:
            print(f"Error reading {gexf_path}: {e}")

    # 履歴をもとに分類
    classification = {}
    
    for agent_id, classes in agent_classes_history.items():
        # クラスが {0, 1} の部分集合 (例: {0}, {1}, または {0, 1}) なら minus-core
        if classes.issubset({0, 1}):
            classification[agent_id] = "minus-core"
            
        # クラスが {3, 4} の部分集合 なら plus-core
        elif classes.issubset({3, 4}):
            classification[agent_id] = "plus-core"
            
        # それ以外 (2が含まれている、あるいは両極端を行き来している) なら noncore
        else:
            classification[agent_id] = "noncore"

    return classification

# ==========================================
# 実行例 (コメントアウトしています)
# ==========================================
# RESULTS_DIR = "./results"
# MAX_INDEX = 40000
# results = identify_core(seed=0, target_sign=0.0)
# print("Sample classification:")
# for agent, role in list(results.items())[:10]:
#     print(f"Agent {agent}: {role}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

def plot_repost_reach_stacked(repost_csv_path, agent_classification, seed, output_dir="./results/LR_diff_summary"):
    """
    リポストのログとエージェントの属性（core/noncore）を統合し、
    指定された順番の100%積み上げグラフを出力する関数。
    """
    print("Loading data and mapping agent classifications...")
    
    # 1. データの読み込み
    df = pd.read_csv(repost_csv_path)

    # 2. agent_classification辞書を用いて、作者とリポスターの属性をマッピング
    df['author_type'] = df['original_author_id'].map(agent_classification)
    df['reposter_type'] = df['reposter_id'].map(agent_classification)

    # 分類できなかったデータ（エラー値など）は除外
    df = df.dropna(subset=['author_type', 'reposter_type'])

    def prepare_plot_data(target_author_type, stack_order):
        # 該当する著者のポストだけを抽出
        target_df = df[df['author_type'] == target_author_type]

        if target_df.empty:
            return None

        # postごとに各reposter_typeの数を集計し、横持ちの表(unstack)に変換
        counts = target_df.groupby(['original_post_id', 'reposter_type']).size().unstack(fill_value=0)

        # 存在しない属性列があった場合、エラーを防ぐため0埋めの列を追加
        for col in stack_order:
            if col not in counts.columns:
                counts[col] = 0

        # 各postの合計repost数を計算
        counts['total_reposts'] = counts.sum(axis=1)

        # 全体のrepost数で降順にソート (最もrepostされたpostが左側に来るように)
        counts = counts.sort_values(by='total_reposts', ascending=False)

        # 各typeの割合(%)を計算 (100%積み上げ用)
        pcts = {}
        for col in stack_order:
            pcts[col] = (counts[col] / counts['total_reposts']) * 100

        return pcts, counts['total_reposts']

    # 3. 図のセットアップ (1行2列の横並び)
    # y軸のスケールを共有(sharey=True)して見やすくします
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

    # 添付画像に合わせたカラー定義
    colors = {
        'minus-core': '#a0a0ff', # 薄い青
        'plus-core': '#ffaaaa',  # 薄い赤
        'noncore': '#e0e0e0'     # 薄いグレー
    }

    # =========================================
    # 左図: author = minus-core のポスト
    # 積み上げ順: minus-core(下) -> plus-core(中) -> noncore(上)
    # =========================================
    order_minus = ['minus-core', 'plus-core', 'noncore']
    data_minus = prepare_plot_data('minus-core', order_minus)
    
    if data_minus is not None:
        pcts_minus, totals_minus = data_minus
        x_minus = np.arange(len(totals_minus))
        y_minus = [pcts_minus[c].values for c in order_minus]
        c_minus = [colors[c] for c in order_minus]

        ax1.stackplot(x_minus, y_minus, labels=order_minus, colors=c_minus, linewidth=0)
        ax1.set_title("Reach of minus-core posts")
        ax1.set_xlim(0, len(x_minus) - 1)
        ax1.set_ylim(0, 100)
        ax1.set_xticks([]) # postが多すぎるためX軸の目盛りは非表示
        
        # Y軸のラベルを 0%, 20%... にフォーマット
        ax1.set_yticks(np.arange(0, 101, 20))
        ax1.set_yticklabels([f"{i}%" for i in range(0, 101, 20)])

    # =========================================
    # 右図: author = plus-core のポスト
    # 積み上げ順: plus-core(下) -> minus-core(中) -> noncore(上)
    # =========================================
    order_plus = ['plus-core', 'minus-core', 'noncore']
    data_plus = prepare_plot_data('plus-core', order_plus)
    
    if data_plus is not None:
        pcts_plus, totals_plus = data_plus
        x_plus = np.arange(len(totals_plus))
        y_plus = [pcts_plus[c].values for c in order_plus]
        c_plus = [colors[c] for c in order_plus]

        ax2.stackplot(x_plus, y_plus, labels=order_plus, colors=c_plus, linewidth=0)
        ax2.set_title("Reach of plus-core posts")
        ax2.set_xlim(0, len(x_plus) - 1)
        ax2.set_ylim(0, 100)
        ax2.set_xticks([])

    # =========================================
    # 凡例とレイアウトの調整
    # =========================================
    # 凡例を下部にまとめて配置
    # 色の設定は共通なので、順序を整えて独自に生成
    handles = [
        plt.Rectangle((0,0),1,1, color=colors['minus-core']),
        plt.Rectangle((0,0),1,1, color=colors['plus-core']),
        plt.Rectangle((0,0),1,1, color=colors['noncore'])
    ]
    labels = ['minus-core accounts', 'plus-core accounts', 'noncore accounts']
    
    fig.legend(handles, labels, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.05), frameon=False, fontsize=12)

    plt.tight_layout()
    
    os.makedirs(output_dir, exist_ok=True)
    out_filename = f"repost_reach_100pct_stacked_seed_{seed}.png"
    out_path = os.path.join(output_dir, out_filename)
    plt.savefig(out_path, bbox_inches='tight', dpi=300)

    print(f"Plot successfully saved to: {out_path}")


# --- 実行時の呼び出しイメージ ---
# 1. まず先ほど完成させた識別関数で辞書を取得
# agent_classification_dict = identify_core(seed=0, target_sign=0.0)
# 
# 2. ログファイルのパスと辞書を渡してグラフ描画関数を呼び出す
# plot_repost_reach_stacked("results/run_0_dir_0.0/repost_log.csv", agent_classification_dict)

In [ ]:
import os
import networkx as nx
import powerlaw
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter, defaultdict

def analyze_and_plot_degree_distribution(gexf_path, agent_classification, seed, output_dir="./results/LR_diff_summary"):    
    """
    指定されたGEXFファイルを読み込み、core/noncoreクラスごとに
    Degree分布のプロット作成とべき乗則フィッティング(Alpha値計算)を行う関数。
    
    Parameters:
        gexf_path (str): 対象のGEXFファイルのパス
        agent_classification (dict): {agent_id: "minus-core"|"plus-core"|"noncore"} の辞書
        output_dir (str): 画像を保存するディレクトリ
        
    Returns:
        dict: 各クラスのべき指数(Alpha)を格納した辞書
    """
    print(f"Loading graph from {gexf_path} ...")
    
    if not os.path.exists(gexf_path):
        print(f"Error: File not found ({gexf_path})")
        return {}

    try:
        G = nx.read_gexf(gexf_path)
    except Exception as e:
        print(f"Error reading GEXF: {e}")
        return {}

    # 辞書のキーを文字列に統一（GEXFのnode IDが文字列であるケースへの対策）
    str_classification = {str(k): v for k, v in agent_classification.items()}

    # クラスごとにDegreeを分類して格納するリスト
    class_degrees = defaultdict(list)
    
    # In-degree ではなく Total Degree (G.degree) を取得
    for node, deg in G.degree():
        # クラスを取得（存在しない場合は除外）
        node_class = str_classification.get(str(node))
        if node_class:
            class_degrees[node_class].append(deg)

    # カラー設定と描画順序
    plot_configs = {
        'minus-core': {'color': 'blue',  'label': 'Minus-Core'},
        'plus-core':  {'color': 'red',   'label': 'Plus-Core'},
        'noncore':    {'color': 'gray',  'label': 'Non-Core'}
    }

    # フィッティング結果を格納する辞書
    alpha_results = {}

    plt.figure(figsize=(9, 6))

    print("Fitting power-law and plotting distributions...")
    
    for cls_name, config in plot_configs.items():
        degrees = class_degrees.get(cls_name, [])
        
        # べき乗則フィットおよびLog-Logプロットのため、次数0は除外
        valid_degrees = [d for d in degrees if d > 0]
        
        if len(valid_degrees) == 0:
            print(f"  [{cls_name}] No valid degrees found (>0).")
            alpha_results[cls_name] = np.nan
            continue

        # 1. べき乗則(Power-law)のフィッティング
        if len(valid_degrees) > 5:
            try:
                fit = powerlaw.Fit(valid_degrees, discrete=True, verbose=False)
                alpha_val = fit.power_law.alpha
                alpha_results[cls_name] = alpha_val
                print(f"  [{cls_name}] Alpha = {alpha_val:.4f} (N={len(valid_degrees)})")
            except Exception as e:
                print(f"  [{cls_name}] Fit failed: {e}")
                alpha_results[cls_name] = np.nan
        else:
            alpha_results[cls_name] = np.nan

        # 2. プロット用データの集計 (各次数の人数をカウント)
        deg_counts = Counter(valid_degrees)
        
        # X軸(次数), Y軸(人数)のリストを作成してソート
        x_vals = sorted(deg_counts.keys())
        y_vals = [deg_counts[x] for x in x_vals]

        # 散布図としてプロット
        plt.scatter(
            x_vals, y_vals, 
            color=config['color'], 
            label=f"{config['label']} ($\\alpha \\approx {alpha_results[cls_name]:.2f}$)", 
            alpha=0.6, 
            s=25,
            edgecolors='white',
            linewidths=0.5
        )

    # プロットの装飾
    step_name = os.path.basename(gexf_path).replace('.gexf', '')
    plt.title(f"Total Degree Distribution by Agent Class ({step_name})", fontsize=15)
    plt.xlabel("Degree ($k$)", fontsize=13)
    plt.ylabel("Number of Nodes (Count)", fontsize=13)
    
    # 両対数(Log-Log)スケールに設定
    plt.xscale('log')
    plt.yscale('log')
    
    plt.grid(True, which="both", linestyle="--", alpha=0.3)
    plt.legend(fontsize=11)
    
    # 保存と表示
    os.makedirs(output_dir, exist_ok=True)
    out_filename = f"degree_dist_by_class_seed_{seed}_{step_name}.png"
    out_path = os.path.join(output_dir, out_filename)
    
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.show()

    print(f"Plot saved to: {out_path}")
    
    return alpha_results


# ==========================================
# 実行例 (コメントアウト)
# ==========================================
# 1. 前の手順で作成した関数でエージェントを分類
# agent_dict = identify_core(seed=0, target_sign=0.0)
#
# 2. 対象となるステップ(例: step_40000)のGEXFパスを指定して実行
# target_gexf = "results/run_0_dir_0.0/GEXF/lambda_0.0/step_40000.gexf"
# alphas = analyze_and_plot_degree_distribution(target_gexf, agent_dict)
#
# print("Fitted Alpha values:", alphas)

In [ ]:
import os
import networkx as nx
import powerlaw
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter, defaultdict

def plot_core_noncore_degree_distribution(gexf_path, agent_classification, seed, output_dir="./results/LR_diff_summary"):
    """
    指定されたGEXFファイルを読み込み、coreエージェント(minus/plus)と
    noncoreエージェント間のエッジのみを考慮した場合の、
    minus-coreとplus-coreのDegree分布とべき乗則(Alpha)を計算してプロットする関数。
    """
    print(f"Loading graph from {gexf_path} ...")
    
    if not os.path.exists(gexf_path):
        print(f"Error: File not found ({gexf_path})")
        return {}

    try:
        G = nx.read_gexf(gexf_path)
    except Exception as e:
        print(f"Error reading GEXF: {e}")
        return {}

    str_classification = {str(k): v for k, v in agent_classification.items()}

    # 各コアエージェントの次数を記録する辞書を初期化
    # { "minus-core": { "node_1": 0, "node_2": 5, ... }, "plus-core": {...} }
    core_degrees = {
        'minus-core': defaultdict(int),
        'plus-core': defaultdict(int)
    }

    # あらかじめグラフ内の全コアエージェントを登録し、初期値0をセット
    for node in G.nodes():
        node_class = str_classification.get(str(node))
        if node_class in core_degrees:
            core_degrees[node_class][node] = 0

    # グラフの全エッジを走査し、core-noncoreの組み合わせのみカウントする
    for u, v in G.edges():
        c_u = str_classification.get(str(u))
        c_v = str_classification.get(str(v))
        
        # uがコアで、vがノンコアの場合
        if c_u in core_degrees and c_v == 'noncore':
            core_degrees[c_u][u] += 1
            
        # vがコアで、uがノンコアの場合
        # (有向グラフで双方向エッジがある場合はそれぞれ独立してカウント=Total Degreeの振る舞い)
        if c_v in core_degrees and c_u == 'noncore':
            core_degrees[c_v][v] += 1

    # カラー設定と描画順序
    plot_configs = {
        'minus-core': {'color': 'blue', 'label': 'Minus-Core'},
        'plus-core':  {'color': 'red',  'label': 'Plus-Core'}
    }

    alpha_results = {}

    plt.figure(figsize=(9, 6))
    print("Fitting power-law and plotting filtered distributions...")
    
    for cls_name, config in plot_configs.items():
        # 記録した次数の値のみをリスト化
        degrees = list(core_degrees[cls_name].values())
        
        # べき乗則フィットおよびLog-Logプロットのため、次数0は除外
        valid_degrees = [d for d in degrees if d > 0]
        
        if len(valid_degrees) == 0:
            print(f"  [{cls_name}] No valid edges to noncore found (>0).")
            alpha_results[cls_name] = np.nan
            continue

        # 1. べき乗則(Power-law)のフィッティング
        if len(valid_degrees) > 5:
            try:
                fit = powerlaw.Fit(valid_degrees, discrete=True, verbose=False)
                alpha_val = fit.power_law.alpha
                alpha_results[cls_name] = alpha_val
                print(f"  [{cls_name}] Alpha = {alpha_val:.4f} (Valid Nodes={len(valid_degrees)})")
            except Exception as e:
                print(f"  [{cls_name}] Fit failed: {e}")
                alpha_results[cls_name] = np.nan
        else:
            alpha_results[cls_name] = np.nan

        # 2. プロット用データの集計 (各次数の人数をカウント)
        deg_counts = Counter(valid_degrees)
        
        x_vals = sorted(deg_counts.keys())
        y_vals = [deg_counts[x] for x in x_vals]

        # 散布図としてプロット
        plt.scatter(
            x_vals, y_vals, 
            color=config['color'], 
            label=f"{config['label']} ($\\alpha \\approx {alpha_results[cls_name]:.2f}$)", 
            alpha=0.7, 
            s=35,
            edgecolors='white',
            linewidths=0.5
        )

    # プロットの装飾
    step_name = os.path.basename(gexf_path).replace('.gexf', '')
    plt.title(f"Degree Distribution of Core Agents (Core-to-Noncore edges only)\n{step_name}", fontsize=14)
    plt.xlabel("Filtered Degree ($k_{core \\leftrightarrow noncore}$)", fontsize=13)
    plt.ylabel("Number of Core Nodes (Count)", fontsize=13)
    
    plt.xscale('log')
    plt.yscale('log')
    
    plt.grid(True, which="both", linestyle="--", alpha=0.3)
    
    # 凡例を表示 (データが存在する場合のみ)
    if alpha_results:
        plt.legend(fontsize=11)
    
    # 保存
    os.makedirs(output_dir, exist_ok=True)
    out_filename = f"filtered_degree_dist_core_noncore_seed_{seed}_{step_name}.png"
    out_path = os.path.join(output_dir, out_filename)
    
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.show()

    print(f"Plot saved to: {out_path}")
    
    return alpha_results

# ==========================================
# 実行例 (コメントアウト)
# ==========================================
# agent_dict = identify_core(seed=0, target_sign=0.0)
# target_gexf = "results/run_0_dir_0.0/GEXF/lambda_0.0/step_40000.gexf"
# alphas = plot_core_noncore_degree_distribution(target_gexf, agent_dict)
# print("Fitted Alpha values:", alphas)

In [ ]:
# ==========================================
# メイン実行ブロック
# ==========================================
if __name__ == "__main__":
    # --- 実行設定 ---
    TARGET_SIGN = -1.0
    SEED_TABLE = [10, 11, 12, 13, 14, 16, 17, 18, 19, 20, 21]
    TARGET_STEP = 40000  # ネットワーク指標を評価するステップ
    
    RESULTS_DIR = "./results"
    OUTPUT_DIR = "./results/LR_diff_summary"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # 全シードのAlpha値を記録しておく辞書（後で平均を出す準備用）
    all_alphas_total = {}
    all_alphas_filtered = {}

    print(f"=== Batch Analysis Started: TARGET_SIGN = {TARGET_SIGN} ===")

    for seed in SEED_TABLE:
        print(f"\n{'='*50}")
        print(f"  Starting analysis for Seed: {seed}")
        print(f"{'='*50}")

        run_dir = os.path.join(RESULTS_DIR, f"run_{seed}_dir_{TARGET_SIGN}")
        if not os.path.isdir(run_dir):
            print(f"Directory not found, skipping: {run_dir}")
            continue

        # 1. エージェントの役割（core/noncore）を特定
        print("[1/4] Identifying core/noncore agents...")
        agent_dict = identify_core(seed=seed, target_sign=TARGET_SIGN)
        if not agent_dict:
            print("  -> Classification failed. Skipping to next seed.")
            continue

        # 2. リポストの伝播割合のスタックプロットを作成
        print("\n[2/4] Generating repost reach stacked plots...")
        repost_csv_path = os.path.join(run_dir, "repost_log.csv")
        if os.path.exists(repost_csv_path):
            plot_repost_reach_stacked(repost_csv_path, agent_dict, seed, output_dir=OUTPUT_DIR)
        else:
            print(f"  -> Repost log not found at {repost_csv_path}")

        # --- 対象ステップのGEXFパスを生成 ---
        target_gexf = os.path.join(run_dir, "GEXF", "lambda_0.0", f"step_{TARGET_STEP}.gexf")
        
        if not os.path.exists(target_gexf):
            print(f"\n[3/4 & 4/4] GEXF file for step {TARGET_STEP} not found. Skipping network analysis.")
            continue

        # 3. クラス別の Total Degree 分布
        print(f"\n[3/4] Analyzing Total Degree distribution (Step: {TARGET_STEP})...")
        alphas_total = analyze_and_plot_degree_distribution(target_gexf, agent_dict, seed, output_dir=OUTPUT_DIR)
        all_alphas_total[seed] = alphas_total

        # 4. Core-Noncore エッジに限定した Degree 分布
        print(f"\n[4/4] Analyzing Core-to-Noncore Degree distribution (Step: {TARGET_STEP})...")
        alphas_filtered = plot_core_noncore_degree_distribution(target_gexf, agent_dict, seed, output_dir=OUTPUT_DIR)
        all_alphas_filtered[seed] = alphas_filtered

    print(f"\n{'='*50}")
    print("=== All Batch Analysis Completed ===")